# Qwen2.5-32B Instruct on Colab

Same **18 generations** as Llama / RTLCoder: 3 circuits × 2 prompts × 3 attempts.

**Model:** `Qwen/Qwen2.5-32B-Instruct` in **4-bit**.
This is **not** ComplexVCoder (no GIR, no RAG, no official weights).

Label slides: **Qwen2.5 32B Instruct (4-bit Colab)**.
If you instead use OpenRouter on ecs05, label **Qwen2.5 32B Instruct**.

## Setup
1. Runtime → Change runtime type → **L4 or A100** (T4 may OOM).
2. Run all cells.
3. Download `qwen2.5-32b-instruct.zip`.
4. On ecs05:

```tcsh
cd /home/ft2335/dataset
unzip -o qwen2.5-32b-instruct.zip
python3 scripts/evaluate_generated.py --model-dir experiments/qwen2.5-32b-instruct
```

Do not edit generated RTL.


In [ ]:
!nvidia-smi -L
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime → Change runtime type → L4 or A100'


In [ ]:
%pip install -q 'transformers>=4.44' accelerate bitsandbytes sentencepiece


In [ ]:
from pathlib import Path

ROOT = Path('/content/cdc_pilot')
PROMPT_DIR = ROOT / 'experiments' / 'prompts'
OUT_ROOT = ROOT / 'experiments' / 'qwen2.5-32b-instruct'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS = {
  "cdc_2phase.functional.md": "Generate synthesizable SystemVerilog for the following module:\n\nmodule cdc_2phase #(\n  parameter WIDTH = 1\n)(\n  input                  src_rst_ni,\n  input                  src_clk_i,\n  input      [WIDTH-1:0] src_data_i,\n  input                  src_valid_i,\n  output                 src_ready_o,\n\n  input                  dst_rst_ni,\n  input                  dst_clk_i,\n  output     [WIDTH-1:0] dst_data_o,\n  output                 dst_valid_o,\n  input                  dst_ready_i\n);\n\nsrc_clk_i and dst_clk_i are independent asynchronous clocks with no fixed\nfrequency or phase relationship.\n\nA source transfer is accepted on a rising src_clk_i edge when src_valid_i and\nsrc_ready_o are both high. Every accepted source item must appear exactly once\nat the destination, in the original order.\n\nA destination transfer completes on a rising dst_clk_i edge when dst_valid_o\nand dst_ready_i are both high. While dst_valid_o is high and dst_ready_i is\nlow, dst_valid_o must remain asserted and dst_data_o must remain stable.\n\nThe design may support one outstanding item. src_ready_o must be low whenever\na new source item cannot safely be accepted.\n\nsrc_rst_ni and dst_rst_ni are active-low resets. Reset must return both\ninterfaces to an idle state and must not create a spurious destination\ntransaction.\n\nPlace the complete implementation, including any helper modules, in one source\nfile. Do not include a testbench, explanation, markdown, vendor primitives, or\nthe reference implementation.\n",
  "cdc_2phase.cdc_explicit.md": "Generate synthesizable SystemVerilog for the following module:\n\nmodule cdc_2phase #(\n  parameter WIDTH = 1\n)(\n  input                  src_rst_ni,\n  input                  src_clk_i,\n  input      [WIDTH-1:0] src_data_i,\n  input                  src_valid_i,\n  output                 src_ready_o,\n\n  input                  dst_rst_ni,\n  input                  dst_clk_i,\n  output     [WIDTH-1:0] dst_data_o,\n  output                 dst_valid_o,\n  input                  dst_ready_i\n);\n\nsrc_clk_i and dst_clk_i are independent asynchronous clocks with no fixed\nfrequency or phase relationship.\n\nA source transfer is accepted on a rising src_clk_i edge when src_valid_i and\nsrc_ready_o are both high. Every accepted source item must appear exactly once\nat the destination, in the original order.\n\nA destination transfer completes on a rising dst_clk_i edge when dst_valid_o\nand dst_ready_i are both high. While dst_valid_o is high and dst_ready_i is\nlow, dst_valid_o must remain asserted and dst_data_o must remain stable.\n\nThe design may support one outstanding item. src_ready_o must be low whenever\na new source item cannot safely be accepted.\n\nsrc_rst_ni and dst_rst_ni are active-low resets. Reset must return both\ninterfaces to an idle state and must not create a spurious destination\ntransaction.\n\nThe implementation must be safe for clock-domain and reset-domain crossings\nand must pass structural CDC/RDC analysis with zero unsafe crossings. Reset\nrelease must be safe in each clock domain, and transferred multi-bit data must\nremain coherent. Select the architecture yourself.\n\nPlace the complete implementation, including any helper modules, in one source\nfile. Do not include a testbench, explanation, markdown, vendor primitives, or\nthe reference implementation.\n",
  "async_fifo.functional.md": "Generate synthesizable Verilog-2001 for this module:\n\nmodule async_fifo #(\n  parameter DSIZE = 8,\n  parameter ASIZE = 4,\n  parameter FALLTHROUGH = \"TRUE\"\n)(\n  input  wire             wclk,\n  input  wire             wrst_n,\n  input  wire             winc,\n  input  wire [DSIZE-1:0] wdata,\n  output wire             wfull,\n  output wire             awfull,\n\n  input  wire             rclk,\n  input  wire             rrst_n,\n  input  wire             rinc,\n  output wire [DSIZE-1:0] rdata,\n  output wire             rempty,\n  output wire             arempty\n);\n\nImplement a FIFO containing 2**ASIZE entries of DSIZE bits. wclk and rclk are\nindependent asynchronous clocks.\n\nA write is accepted on a rising wclk edge when winc is high and wfull is low.\nWrites attempted while full must not alter FIFO contents.\n\nA read is accepted on a rising rclk edge when rinc is high and rempty is low.\nReads attempted while empty must not advance the FIFO.\n\nAccepted data must be returned exactly once and in write order. wfull is\ngenerated in the write domain and rempty in the read domain. awfull indicates\nthat the FIFO is approaching full, and arempty indicates that it is approaching\nempty.\n\nAfter reset, wfull must be low and rempty must be high. wrst_n and rrst_n are\nactive-low resets belonging to their respective clock domains.\n\nWhen FALLTHROUGH equals \"TRUE\", rdata presents the current oldest unread word\nwithout requiring an additional registered-read cycle. Otherwise, rdata may be\nupdated by an accepted read.\n\nPlace the complete implementation and helper modules in one file. Do not\ninclude a testbench, explanation, markdown, vendor primitives, or reference\ncode.\n",
  "async_fifo.cdc_explicit.md": "Generate synthesizable Verilog-2001 for this module:\n\nmodule async_fifo #(\n  parameter DSIZE = 8,\n  parameter ASIZE = 4,\n  parameter FALLTHROUGH = \"TRUE\"\n)(\n  input  wire             wclk,\n  input  wire             wrst_n,\n  input  wire             winc,\n  input  wire [DSIZE-1:0] wdata,\n  output wire             wfull,\n  output wire             awfull,\n\n  input  wire             rclk,\n  input  wire             rrst_n,\n  input  wire             rinc,\n  output wire [DSIZE-1:0] rdata,\n  output wire             rempty,\n  output wire             arempty\n);\n\nImplement a FIFO containing 2**ASIZE entries of DSIZE bits. wclk and rclk are\nindependent asynchronous clocks.\n\nA write is accepted on a rising wclk edge when winc is high and wfull is low.\nWrites attempted while full must not alter FIFO contents.\n\nA read is accepted on a rising rclk edge when rinc is high and rempty is low.\nReads attempted while empty must not advance the FIFO.\n\nAccepted data must be returned exactly once and in write order. wfull is\ngenerated in the write domain and rempty in the read domain. awfull indicates\nthat the FIFO is approaching full, and arempty indicates that it is approaching\nempty.\n\nAfter reset, wfull must be low and rempty must be high. wrst_n and rrst_n are\nactive-low resets belonging to their respective clock domains.\n\nWhen FALLTHROUGH equals \"TRUE\", rdata presents the current oldest unread word\nwithout requiring an additional registered-read cycle. Otherwise, rdata may be\nupdated by an accepted read.\n\nThe implementation must be safe for clock-domain and reset-domain crossings\nand must pass structural CDC/RDC analysis with zero unsafe crossings. Reset\nrelease must be safe in each clock domain, and transferred multi-bit data must\nremain coherent. Select the architecture yourself.\n\nPlace the complete implementation and helper modules in one file. Do not\ninclude a testbench, explanation, markdown, vendor primitives, or reference\ncode.\n",
  "apbxclk.functional.md": "Generate synthesizable Verilog-2001 implementing an APB clock-domain bridge:\n\nmodule apbxclk #(\n  parameter C_APB_ADDR_WIDTH = 12,\n  parameter C_APB_DATA_WIDTH = 32,\n  parameter [0:0] OPT_REGISTERED = 1'b0\n)(\n  input  wire                         S_APB_PCLK,\n  input  wire                         S_PRESETn,\n  input  wire                         S_APB_PSEL,\n  input  wire                         S_APB_PENABLE,\n  output reg                          S_APB_PREADY,\n  input  wire [C_APB_ADDR_WIDTH-1:0]  S_APB_PADDR,\n  input  wire                         S_APB_PWRITE,\n  input  wire [C_APB_DATA_WIDTH-1:0]  S_APB_PWDATA,\n  input  wire [C_APB_DATA_WIDTH/8-1:0] S_APB_PWSTRB,\n  input  wire [2:0]                   S_APB_PPROT,\n  output wire [C_APB_DATA_WIDTH-1:0]  S_APB_PRDATA,\n  output wire                         S_APB_PSLVERR,\n\n  input  wire                         M_APB_PCLK,\n  output reg                          M_PRESETn,\n  output reg                          M_APB_PSEL,\n  output reg                          M_APB_PENABLE,\n  input  wire                         M_APB_PREADY,\n  output wire [C_APB_ADDR_WIDTH-1:0]  M_APB_PADDR,\n  output wire                         M_APB_PWRITE,\n  output wire [C_APB_DATA_WIDTH-1:0]  M_APB_PWDATA,\n  output wire [C_APB_DATA_WIDTH/8-1:0] M_APB_PWSTRB,\n  output wire [2:0]                   M_APB_PPROT,\n  input  wire [C_APB_DATA_WIDTH-1:0]  M_APB_PRDATA,\n  input  wire                         M_APB_PSLVERR\n);\n\nS_APB_PCLK and M_APB_PCLK are independent asynchronous clocks.\n\nAccept standard APB transfers on the S_APB interface. Forward each accepted\nrequest exactly once to the M_APB interface using a normal APB setup phase\nfollowed by an access phase. Keep the downstream request fields stable until\nM_APB_PREADY completes the transfer.\n\nReturn read data and slave-error status to the source interface. Assert\nS_APB_PREADY only when the corresponding downstream transaction has completed.\nDo not lose, duplicate, or reorder requests. Supporting one outstanding\ntransaction is sufficient.\n\nS_PRESETn is the active-low source reset. M_PRESETn is an active-low reset\noutput for the destination domain. Both interfaces must remain inactive during\nreset, and reset must not create a transaction.\n\nOPT_REGISTERED selects whether crossing payload and response fields are\nexplicitly registered. Both parameter settings must preserve APB behavior.\n\nReturn one complete source file only. Do not include a testbench, explanation,\nmarkdown, vendor primitives, or reference code.\n",
  "apbxclk.cdc_explicit.md": "Generate synthesizable Verilog-2001 implementing an APB clock-domain bridge:\n\nmodule apbxclk #(\n  parameter C_APB_ADDR_WIDTH = 12,\n  parameter C_APB_DATA_WIDTH = 32,\n  parameter [0:0] OPT_REGISTERED = 1'b0\n)(\n  input  wire                         S_APB_PCLK,\n  input  wire                         S_PRESETn,\n  input  wire                         S_APB_PSEL,\n  input  wire                         S_APB_PENABLE,\n  output reg                          S_APB_PREADY,\n  input  wire [C_APB_ADDR_WIDTH-1:0]  S_APB_PADDR,\n  input  wire                         S_APB_PWRITE,\n  input  wire [C_APB_DATA_WIDTH-1:0]  S_APB_PWDATA,\n  input  wire [C_APB_DATA_WIDTH/8-1:0] S_APB_PWSTRB,\n  input  wire [2:0]                   S_APB_PPROT,\n  output wire [C_APB_DATA_WIDTH-1:0]  S_APB_PRDATA,\n  output wire                         S_APB_PSLVERR,\n\n  input  wire                         M_APB_PCLK,\n  output reg                          M_PRESETn,\n  output reg                          M_APB_PSEL,\n  output reg                          M_APB_PENABLE,\n  input  wire                         M_APB_PREADY,\n  output wire [C_APB_ADDR_WIDTH-1:0]  M_APB_PADDR,\n  output wire                         M_APB_PWRITE,\n  output wire [C_APB_DATA_WIDTH-1:0]  M_APB_PWDATA,\n  output wire [C_APB_DATA_WIDTH/8-1:0] M_APB_PWSTRB,\n  output wire [2:0]                   M_APB_PPROT,\n  input  wire [C_APB_DATA_WIDTH-1:0]  M_APB_PRDATA,\n  input  wire                         M_APB_PSLVERR\n);\n\nS_APB_PCLK and M_APB_PCLK are independent asynchronous clocks.\n\nAccept standard APB transfers on the S_APB interface. Forward each accepted\nrequest exactly once to the M_APB interface using a normal APB setup phase\nfollowed by an access phase. Keep the downstream request fields stable until\nM_APB_PREADY completes the transfer.\n\nReturn read data and slave-error status to the source interface. Assert\nS_APB_PREADY only when the corresponding downstream transaction has completed.\nDo not lose, duplicate, or reorder requests. Supporting one outstanding\ntransaction is sufficient.\n\nS_PRESETn is the active-low source reset. M_PRESETn is an active-low reset\noutput for the destination domain. Both interfaces must remain inactive during\nreset, and reset must not create a transaction.\n\nOPT_REGISTERED selects whether crossing payload and response fields are\nexplicitly registered. Both parameter settings must preserve APB behavior.\n\nThe implementation must be safe for clock-domain and reset-domain crossings\nand must pass structural CDC/RDC analysis with zero unsafe crossings. Reset\nrelease must be safe in each clock domain, and transferred multi-bit data must\nremain coherent. Select the architecture yourself.\n\nReturn one complete source file only. Do not include a testbench, explanation,\nmarkdown, vendor primitives, or reference code.\n"
}
for name, text in PROMPTS.items():
    (PROMPT_DIR / name).write_text(text)
print('wrote', len(PROMPTS), 'prompts')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, time, json, re
from datetime import datetime, timezone
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-32B-Instruct'
TEMPERATURE = 0.2
TOP_P = 0.95
MAX_NEW_TOKENS = 4096
CIRCUITS = ['cdc_2phase', 'async_fifo', 'apbxclk']
PROMPT_TYPES = ['functional', 'cdc_explicit']

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('loaded', MODEL_ID, '4-bit')


In [ ]:
def extract_verilog(text: str) -> str:
    blocks = re.findall(r'```(?:systemverilog|verilog|sv|v)?\s*(.*?)```', text, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip() + '\n'
    return text.strip() + '\n'


def next_attempt(base: Path) -> Path:
    existing = [
        int(p.name.split('-')[1])
        for p in base.glob('attempt-*')
        if p.name.split('-')[-1].isdigit()
    ]
    return base / f'attempt-{max(existing, default=0) + 1:03d}'


def generate_one(circuit: str, prompt_type: str):
    prompt_path = PROMPT_DIR / f'{circuit}.{prompt_type}.md'
    prompt = prompt_path.read_text()
    out_dir = next_attempt(OUT_ROOT / circuit / prompt_type)
    out_dir.mkdir(parents=True, exist_ok=False)
    (out_dir / 'generated').mkdir()
    (out_dir / 'prompt.md').write_text(prompt)
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        sample = model.generate(
            inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    text = tokenizer.decode(sample[0][inputs.shape[-1]:], skip_special_tokens=True)
    (out_dir / 'response.txt').write_text(text if text.endswith('\n') else text + '\n')
    (out_dir / 'generated' / f'{circuit}.v').write_text(extract_verilog(text))
    meta = {
        'provider': 'colab-gpu',
        'model': MODEL_ID,
        'load_dtype': 'int4',
        'circuit': circuit,
        'prompt_type': prompt_type,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_new_tokens': MAX_NEW_TOKENS,
        'elapsed_s': round(elapsed, 3),
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'gpu': torch.cuda.get_device_name(0),
        'note': 'Qwen2.5-32B-Instruct 4-bit. Same prompts as Llama. Not ComplexVCoder.',
    }
    (out_dir / 'metadata.json').write_text(json.dumps(meta, indent=2) + '\n')
    print(f'{circuit}/{prompt_type}/{out_dir.name}  {elapsed:.1f}s')
    return out_dir

jobs = []
for circuit in CIRCUITS:
    for prompt_type in PROMPT_TYPES:
        have = len(list((OUT_ROOT / circuit / prompt_type).glob('attempt-*')))
        jobs.extend([(circuit, prompt_type)] * max(0, 3 - have))
print('jobs remaining', len(jobs))
for circuit, prompt_type in jobs:
    generate_one(circuit, prompt_type)
print('done')


In [ ]:
from google.colab import files
import shutil
n = len(list(OUT_ROOT.glob('*/*/attempt-*/generated/*.v')))
print('generated files', n)
assert n == 18, f'expected 18 Verilog files, got {n}'
zip_path = shutil.make_archive('/content/qwen2.5-32b-instruct', 'zip', ROOT, 'experiments/qwen2.5-32b-instruct')
print(zip_path)
files.download(zip_path)
